# Frequency-dependent reflection coefficients

A real acoustic termination — a perforated liner, a Helmholtz resonator, an area change, a
measured impedance — reflects differently at every frequency. Nefes lets a terminal's
`PerturbationBC` carry a reflection coefficient $R$ that is

* a **constant** complex number,
* a **table** `(freqs_hz, values)` interpolated in frequency, or
* a **callable** `R(freq_hz) -> complex` (a frequency-domain model).

This short notebook drives a duct terminated by a frequency-dependent $R(f)$, reads the input
reflection, and verifies it against transmission-line theory: a wave launched at the inlet
reflects off the termination and returns with the round-trip phase, so the **measured input
reflection** is

$$ \Gamma_\text{in}(\omega) \;=\; R(\omega)\, e^{-i\omega(\tau_+ + \tau_-)}, \qquad
   \tau_\pm = \frac{L}{c \pm u}. $$

In [ ]:
import os, sys

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from nefes.elements import catalog as cat
from nefes.perturbation.operator.boundary_bc import PerturbationBC
from nefes.perturbation import forced_response
from nefes.plotting import use_nefes_theme
from nefes.solver import solve
from nefes.solver.control import states_table
from nefes.thermo.configure import perfect_gas
from nefes.assembly.derive import ES_U, ES_C

use_nefes_theme()
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)

## 1. A frequency-dependent termination model

We use a damped single-resonance reflection — a textbook resonator response: $|R|$ peaks near a
tuned frequency and the phase swings through it. Any callable would do; this one is
representative of a tuned liner / quarter-wave element.

In [ ]:
f0, zeta, R0 = 420.0, 0.18, 0.85   # resonance frequency [Hz], damping, peak reflection

def R_of_f(f):
    "Damped single-resonance reflection coefficient R(f) (a callable boundary model)."
    s = 1j * f / f0
    return R0 / (1.0 + 2.0 * zeta * s + s**2)

fshow = np.linspace(20.0, 1200.0, 400)
Rshow = np.array([R_of_f(f) for f in fshow])
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("|R(f)|", "phase R(f)"))
fig.add_scatter(x=fshow, y=np.abs(Rshow), mode="lines", row=1, col=1)
fig.add_scatter(x=fshow, y=np.degrees(np.angle(Rshow)), mode="lines", row=2, col=1)
fig.update_yaxes(title_text="|R|", row=1, col=1)
fig.update_yaxes(title_text="deg", row=2, col=1)
fig.update_xaxes(title_text="frequency [Hz]", row=2, col=1)
fig.update_layout(title="Prescribed frequency-dependent reflection coefficient", showlegend=False)
fig.show()

## 2. Terminate a duct with it and measure the input reflection

A total-pressure inlet drives a unit acoustic wave; a `pressure_outlet` carries the
frequency-dependent reflection as its `PerturbationBC`. We solve the forced response over a
sweep and read $\Gamma_\text{in} = g/f$ at the inlet edge, then compare to
$R(f)\,e^{-i\omega(\tau_++\tau_-)}$.

In [ ]:
L, AREA = 0.5, 0.05
els = [
    cat.total_pressure_inlet(108000.0, 300.0, name="driver", perturbation_bc=PerturbationBC.anechoic(driven=("acoustic",))),
    cat.duct(L, name="duct"),
    cat.pressure_outlet(101325.0, 300.0, name="liner", perturbation_bc=PerturbationBC.reflection(R_of_f)),
]
prob = cat.build_problem(perfect_gas(R_AIR, GAMMA), els, [(0, 1, AREA), (1, 2, AREA)], 5.0, 1e5, CP * 300.0)
res = solve(prob)
assert res.converged
est = states_table(prob, res.x)
u, c = float(est[ES_U, 0]), float(est[ES_C, 0])

freqs = np.linspace(20.0, 1200.0, 200)
fr = forced_response(prob, res.x, freqs)
gamma_in = fr.reflection_at(0)
omega = 2.0 * np.pi * freqs
expected = np.array([R_of_f(f) for f in freqs]) * np.exp(-1j * omega * (L / (u + c) + L / (c - u)))
print(f"mean flow in the duct: u = {u:.1f} m/s, c = {c:.1f} m/s (M = {u/c:.3f})")
print(f"max |measured - R(f)*roundtrip| = {np.max(np.abs(gamma_in - expected)):.2e}")

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("|Gamma_in|", "phase Gamma_in"))
fig.add_scatter(x=freqs, y=np.abs(gamma_in), mode="lines", name="measured", row=1, col=1)
fig.add_scatter(x=freqs, y=np.abs(expected), mode="lines", line=dict(dash="dash"),
                name="R(f) x roundtrip", row=1, col=1)
fig.add_scatter(x=freqs, y=np.degrees(np.angle(gamma_in)), mode="lines", name="measured", row=2, col=1)
fig.add_scatter(x=freqs, y=np.degrees(np.angle(expected)), mode="lines", line=dict(dash="dash"),
                name="R(f) x roundtrip", row=2, col=1)
fig.update_yaxes(title_text="|Gamma_in|", row=1, col=1)
fig.update_yaxes(title_text="deg", row=2, col=1)
fig.update_xaxes(title_text="frequency [Hz]", row=2, col=1)
fig.update_layout(title="Measured input reflection vs transmission-line prediction")
fig.show()

The measured input reflection tracks $R(f)\,e^{-i\omega(\tau_++\tau_-)}$ to round-off: the
prescribed frequency-dependent boundary is reproduced exactly, with the duct round-trip phase on
top. (The resonance in $|R|$ shows through, modulated by the duct's own standing-wave pattern.)

## 3. The table form interpolates to the same thing

Sampling $R(f)$ onto a coarse table and passing `reflection((freqs, values))` gives a linearly
interpolated $R$ — handy for measured data. On a grid that resolves the resonance it matches the
callable closely.

In [ ]:
f_tab = np.linspace(0.0, 1400.0, 71)            # coarse table in Hz
R_tab = np.array([R_of_f(f) for f in f_tab])
els[2] = cat.pressure_outlet(101325.0, 300.0, name="liner", perturbation_bc=PerturbationBC.reflection((f_tab, R_tab)))
prob_t = cat.build_problem(perfect_gas(R_AIR, GAMMA), els, [(0, 1, AREA), (1, 2, AREA)], 5.0, 1e5, CP * 300.0)
res_t = solve(prob_t)
fr_t = forced_response(prob_t, res_t.x, freqs)
print(f"callable vs table input reflection: max diff = {np.max(np.abs(fr_t.reflection_at(0) - gamma_in)):.2e}")
fig = go.Figure()
fig.add_scatter(x=freqs, y=np.abs(gamma_in), mode="lines", name="callable R(f)")
fig.add_scatter(x=freqs, y=np.abs(fr_t.reflection_at(0)), mode="lines", line=dict(dash="dot"),
                name="table (interpolated)")
fig.update_layout(title="Callable vs tabulated reflection coefficient",
                  xaxis_title="frequency [Hz]", yaxis_title="|Gamma_in|")
fig.show()

## Summary

* A terminal's reflection coefficient may be a constant, a `(freqs, values)` table, or a callable
  `R(freq_hz)` — all via `PerturbationBC.reflection(...)`.
* The forced response reproduces the prescribed $R(f)$ exactly: the measured input reflection is
  $R(f)\,e^{-i\omega(\tau_++\tau_-)}$ to round-off (§2), and the table form interpolates to the
  same result (§3).
* Frequencies are in **Hz** throughout (project convention), matching the public perturbation API.